# Módulo 10: Detección de Objetos con Deep Learning (`cv2.dnn`)

Hasta el módulo 8 usamos detectores clásicos (Haar Cascades, HOG, DeepFace) que están entrenados para una tarea muy específica (caras, personas) y funcionan bien pero con margen limitado. En este módulo damos un paso más: usamos `cv2.dnn`, el módulo de OpenCV para cargar y correr redes neuronales ya entrenadas (Deep Learning), sin necesidad de instalar frameworks pesados como PyTorch o TensorFlow.

Vamos a usar **YOLOv4-tiny** (formato Darknet), una red liviana entrenada sobre el dataset **COCO** (80 clases: persona, auto, perro, celular, etc.), que detecta múltiples objetos en una sola pasada y es lo suficientemente rápida como para correr en tiempo real con webcam.

### Diferencia con los detectores clásicos del módulo 8
- Los clásicos (Haar Cascades, HOG) buscan patrones específicos predefinidos (ej. rostros) usando features hechas a mano.
- YOLO es una red neuronal convolucional que aprendió a detectar 80 clases distintas de objetos a la vez, con mejor precisión y generalización.
- `cv2.dnn` no entrena redes, solo las carga y ejecuta (inferencia): la arquitectura (`.cfg`) y los pesos (`.weights`) ya vienen entrenados.

### Antes de empezar
Necesitás los archivos del modelo en `../modelos/`. Corré esto **una sola vez**, desde la raíz del repo:

```bash
python scripts/descargar_modelos.py
```

Esto descarga `yolov4-tiny.cfg`, `yolov4-tiny.weights` y `coco.names` a la carpeta `modelos/`.

## Carga del modelo

Cargamos la red con `cv2.dnn.readNetFromDarknet`, pasando el archivo de configuración (`.cfg`) y los pesos (`.weights`). También leemos los nombres de las 80 clases de COCO desde `coco.names` (un nombre de clase por línea) y obtenemos los nombres de las capas de salida de la red, que son las que necesitamos para leer las detecciones.

In [ ]:
import cv2  # Módulo principal de OpenCV, incluye el submódulo cv2.dnn para cargar y ejecutar redes neuronales ya entrenadas
import numpy as np  # NumPy para manejar los arrays de detecciones/cajas que devuelve la red

net = cv2.dnn.readNetFromDarknet('../modelos/yolov4-tiny.cfg', '../modelos/yolov4-tiny.weights')  # Carga la arquitectura de la red (.cfg, define las capas y su conexión) y los pesos ya entrenados (.weights) en formato Darknet, el framework original en el que se entrenó YOLO

with open('../modelos/coco.names', 'r') as archivo:
    clases = [linea.strip() for linea in archivo.readlines()]  # Lee el archivo de nombres de clases de COCO (una clase por línea) y arma una lista donde el índice coincide con el ID de clase que devuelve la red

output_layers = net.getUnconnectedOutLayersNames()  # Obtiene los nombres de las capas de salida de la red (las capas "no conectadas" hacia adelante, es decir las finales); son las que hay que pedirle a net.forward() para leer las detecciones, porque YOLO tiene varias cabezas de salida a distintas escalas

print(f"Modelo cargado. Cantidad de clases: {len(clases)}")
print(f"Capas de salida: {output_layers}")

## Detección sobre una imagen estática

El pipeline de `cv2.dnn` para hacer inferencia siempre sigue estos pasos:

1. Convertir la imagen en un **blob** con `cv2.dnn.blobFromImage`: redimensiona a 416x416 (tamaño de entrada esperado por YOLOv4-tiny), normaliza los píxeles (factor `1/255`) y convierte de BGR a RGB (`swapRB=True`, porque OpenCV lee en BGR pero la red espera RGB).
2. Pasar el blob a la red con `net.setInput(blob)`.
3. Ejecutar la inferencia con `net.forward(output_layers)`, que devuelve las detecciones crudas.
4. Filtrar las detecciones por confianza mínima y aplicar **Non-Max Suppression** (`cv2.dnn.NMSBoxes`) para eliminar cajas duplicadas que apuntan al mismo objeto.
5. Dibujar las cajas y etiquetas sobre la imagen original.

In [ ]:
imagen = cv2.imread('../imgs/img1.jpg')  # Carga la imagen en BGR (formato por defecto de OpenCV)
alto, ancho = imagen.shape[:2]  # Guarda las dimensiones originales: las coordenadas que devuelve la red vienen normalizadas (0-1) y hay que multiplicarlas por estas dimensiones para volver a píxeles reales

blob = cv2.dnn.blobFromImage(imagen, scalefactor=1/255, size=(416, 416), swapRB=True, crop=False)  # Convierte la imagen en un "blob" (tensor 4D) apto para la red: redimensiona a 416x416 porque YOLOv4-tiny fue entrenada con ese tamaño de entrada fijo; scalefactor=1/255 normaliza los píxeles de [0,255] a [0,1] porque la red espera valores en ese rango; swapRB=True intercambia los canales BGR->RGB porque OpenCV lee en BGR pero la red fue entrenada con imágenes en RGB. Si se omite cualquiera de estos pasos la red produce detecciones incorrectas o directamente falla
net.setInput(blob)  # Carga el blob como entrada de la red para la siguiente pasada hacia adelante
detecciones = net.forward(output_layers)  # Ejecuta la inferencia (forward pass) y devuelve las detecciones crudas de cada capa de salida: por cada celda de la grilla, para cada anchor box, un vector [centro_x, centro_y, w, h, objectness, clase_0, clase_1, ...] normalizado entre 0 y 1

CONFIANZA_MINIMA = 0.5  # Umbral de confianza: solo se conservan detecciones donde la red está razonablemente segura de la clase, para filtrar ruido

cajas = []
confianzas = []
ids_clases = []

for salida in detecciones:  # Recorre las detecciones de cada capa de salida (YOLO tiene varias escalas de detección)
    for deteccion in salida:  # Recorre cada detección individual (una por anchor box en cada celda de la grilla)
        puntajes = deteccion[5:]  # Los primeros 4 valores son la caja (centro_x, centro_y, w, h), el 5to es "objectness"; a partir del índice 5 vienen los puntajes de probabilidad por cada una de las 80 clases de COCO
        id_clase = np.argmax(puntajes)  # Índice de la clase con mayor puntaje: la clase más probable para esta detección
        confianza = puntajes[id_clase]  # Puntaje (probabilidad) de esa clase más probable

        if confianza > CONFIANZA_MINIMA:  # Descarta detecciones poco confiables antes de seguir procesando
            centro_x = int(deteccion[0] * ancho)  # Desnormaliza el centro X: la red devuelve coordenadas relativas (0-1) al tamaño de entrada, se multiplican por el ancho real de la imagen para volver a píxeles
            centro_y = int(deteccion[1] * alto)  # Ídem para el centro Y, usando el alto real
            w = int(deteccion[2] * ancho)  # Ancho de la caja detectada, desnormalizado
            h = int(deteccion[3] * alto)  # Alto de la caja detectada, desnormalizado

            x = int(centro_x - w / 2)  # YOLO devuelve el centro de la caja, pero cv2.rectangle necesita la esquina superior izquierda: se resta la mitad del ancho
            y = int(centro_y - h / 2)  # Ídem para la coordenada Y: se resta la mitad del alto para obtener la esquina superior

            cajas.append([x, y, w, h])
            confianzas.append(float(confianza))
            ids_clases.append(id_clase)

indices = cv2.dnn.NMSBoxes(cajas, confianzas, CONFIANZA_MINIMA, nms_threshold=0.4)  # Non-Max Suppression: cuando varios anchor boxes detectan el mismo objeto se generan cajas superpuestas y redundantes; NMSBoxes se queda con la caja de mayor confianza y descarta las demás que se solapan por encima de nms_threshold (0.4 = 40% de superposición/IoU). Es necesario aplicarlo después del filtro de confianza porque ese filtro por sí solo no elimina duplicados, solo descarta detecciones de baja probabilidad

print(f"Detecciones antes de NMS: {len(cajas)}")
print(f"Detecciones después de NMS: {len(indices)}")

In [ ]:
imagen_resultado = imagen.copy()  # Copia para dibujar encima sin modificar la imagen original

for i in indices:  # `indices` contiene los índices (dentro de `cajas`/`confianzas`/`ids_clases`) que sobrevivieron al filtro de NMS
    x, y, w, h = cajas[i]
    etiqueta = f"{clases[ids_clases[i]]}: {confianzas[i]:.2f}"  # Arma el texto a mostrar: nombre de la clase (buscado por índice en la lista `clases`) más la confianza con 2 decimales

    cv2.rectangle(imagen_resultado, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Dibuja el bounding box de la detección en verde (BGR)
    cv2.putText(imagen_resultado, etiqueta, (x, max(y - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)  # Escribe la etiqueta arriba del box; max(y - 10, 0) evita que el texto se salga de la imagen si el box está pegado al borde superior

cv2.imshow('Detección de Objetos - Imagen', imagen_resultado)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Detección en tiempo real con webcam

Mismo pipeline (blob → `setInput` → `forward` → filtrado + NMS → dibujo), pero corriendo dentro de un loop sobre cada frame de la webcam. Presioná `q` para salir.

In [ ]:
CONFIANZA_MINIMA = 0.5  # Mismo umbral de confianza que en la detección estática

vid = cv2.VideoCapture(0)  # Abre la cámara web por defecto (índice 0) como fuente de video
cv2.namedWindow('Detección de Objetos - Webcam', cv2.WINDOW_NORMAL)  # Crea la ventana como redimensionable por el usuario, en vez del tamaño fijo por defecto

while True:
    ret, frame = vid.read()  # Lee un frame de la webcam; `ret` indica si la lectura fue exitosa
    if not ret:
        break  # Corta el loop si falla la captura (por ejemplo, la cámara se desconectó)

    alto, ancho = frame.shape[:2]  # Dimensiones del frame actual, necesarias para desnormalizar las coordenadas de las detecciones

    blob = cv2.dnn.blobFromImage(frame, scalefactor=1/255, size=(416, 416), swapRB=True, crop=False)  # Mismo preprocesamiento que en la imagen estática: redimensiona a 416x416, normaliza a [0,1] y convierte BGR->RGB, ahora aplicado a cada frame del video
    net.setInput(blob)
    detecciones = net.forward(output_layers)  # Corre la inferencia sobre el frame actual; se repite en cada iteración porque cada frame es una imagen distinta

    cajas = []
    confianzas = []
    ids_clases = []

    for salida in detecciones:
        for deteccion in salida:
            puntajes = deteccion[5:]  # Puntajes de las 80 clases de COCO para esta detección (a partir del índice 5, después de la caja y el objectness)
            id_clase = np.argmax(puntajes)  # Clase con mayor puntaje
            confianza = puntajes[id_clase]  # Puntaje de esa clase

            if confianza > CONFIANZA_MINIMA:  # Filtra detecciones poco confiables antes de desnormalizar coordenadas
                centro_x = int(deteccion[0] * ancho)  # Desnormaliza el centro X multiplicando por el ancho real del frame
                centro_y = int(deteccion[1] * alto)  # Desnormaliza el centro Y multiplicando por el alto real del frame
                w = int(deteccion[2] * ancho)  # Ancho de la caja desnormalizado
                h = int(deteccion[3] * alto)  # Alto de la caja desnormalizado

                x = int(centro_x - w / 2)  # Convierte de (centro, tamaño) a esquina superior izquierda, formato que espera cv2.rectangle
                y = int(centro_y - h / 2)  # Ídem para Y

                cajas.append([x, y, w, h])
                confianzas.append(float(confianza))
                ids_clases.append(id_clase)

    indices = cv2.dnn.NMSBoxes(cajas, confianzas, CONFIANZA_MINIMA, nms_threshold=0.4)  # Non-Max Suppression: descarta cajas duplicadas que apuntan al mismo objeto, necesario en cada frame igual que en la imagen estática

    for i in indices:
        x, y, w, h = cajas[i]
        etiqueta = f"{clases[ids_clases[i]]}: {confianzas[i]:.2f}"

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Dibuja el box directamente sobre el frame (no se copia, se muestra al toque)
        cv2.putText(frame, etiqueta, (x, max(y - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow('Detección de Objetos - Webcam', frame)  # Muestra el frame ya anotado con las cajas y etiquetas

    if cv2.waitKey(1) == ord('q'):  # Espera 1ms por una tecla (necesario para refrescar la ventana en cada iteración); si es 'q', corta el loop
        break

vid.release()  # Libera el dispositivo de la webcam para que otros programas puedan usarla
cv2.destroyAllWindows()  # Cierra la ventana de OpenCV

## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/10_practica.ipynb`](../practicas/10_practica.ipynb).